# S6-02: 구조공학 자동화 — 스크립트 리뷰, 시방서 요약, 계약서 비교
**Skilljar S6 응용: Claude Code + 건설 문서 분석 (실라버스 W9)**

## 학습 목표
- Claude API를 사용하여 구조 계산 스크립트를 자동 리뷰할 수 있다
- 건설 시방서를 구조화된 JSON으로 요약할 수 있다
- 계약서의 주요 조항을 비교 분석하여 위험 요소를 식별할 수 있다

## 사전 준비
1. `.env` 파일에 `ANTHROPIC_API_KEY` 설정
2. 이전 실습(S6_01)에서 배운 Claude API 호출 패턴 숙지

> **참고**: 이 노트북의 모든 예제는 **구조공학 도메인**에 특화되어 있습니다.
> Claude Code에서 이 코드들을 직접 생성하고 테스트하는 것이 최종 목표입니다.

In [ ]:
# 패키지 설치
%pip install anthropic python-dotenv

In [ ]:
# 환경변수 로드 및 클라이언트 생성
import json
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-0"

---
## Exercise 1: 구조 계산 스크립트 자동 리뷰

### 배경
구조 엔지니어가 작성한 Python 스크립트에는 종종 다음과 같은 문제가 있습니다:
- 단위 불일치 (mm와 m 혼용)
- KDS 조항 번호 누락
- 입력값 검증 부재
- 타입 힌트/독스트링 미작성

Claude Code는 이러한 코드를 자동으로 리뷰하고 개선합니다.
이 실습에서는 Claude API를 사용하여 리뷰 시스템을 구현합니다.

### 문제
아래에 **의도적으로 문제가 있는** RC 보 설계 스크립트가 제공됩니다.
Claude API를 사용하여 이 코드를 리뷰하고, 개선된 버전을 생성하세요.

### 요구사항
1. `review_structural_code()` 함수: 코드를 분석하여 문제점과 개선사항을 JSON으로 반환
2. `improve_structural_code()` 함수: 문제를 수정한 개선 코드를 반환
3. 리뷰 결과에 KDS 조항 준수 여부를 포함

### 기대 출력
```json
{
  "issues": [
    {"severity": "critical", "line": "...", "description": "..."},
    ...
  ],
  "kds_compliance": {
    "clause_references": false,
    "unit_consistency": false,
    "phi_parametrized": false
  },
  "overall_score": 3
}
```

In [ ]:
# ===== Exercise 1: 구조 스크립트 자동 리뷰 =====

# 리뷰 대상: 의도적으로 문제가 있는 코드
buggy_code = '''
def calc_beam(b, d, fc, fy, Mu):
    # 등가 압축 깊이 계산
    a = Mu / (0.85 * fc * b * (d - Mu / (2 * 0.85 * fc * b * d)))
    As = Mu / (fy * (d - a/2))
    rho = As / (b * d)
    
    # 최소 철근비
    rho_min = max(0.25 * fc**0.5 / fy, 1.4/fy)
    
    # 전단 검토
    Vc = 0.16 * fc**0.5 * b * d  # 이 공식이 맞나?
    
    if rho >= rho_min:
        print("OK")
    else:
        print("NG")
    
    return As, rho, Vc
'''.strip()

print("=== 리뷰 대상 코드 ===")
print(buggy_code)

In [ ]:
# Step 1: 코드 리뷰 함수
def review_structural_code(code: str) -> dict:
    """구조 계산 Python 코드를 리뷰하여 문제점을 분석

    Args:
        code: 리뷰할 Python 소스 코드

    Returns:
        dict: 리뷰 결과 {issues, kds_compliance, overall_score}
    """
    system = """당신은 KDS 콘크리트구조 설계기준에 정통한 구조공학 코드 리뷰어입니다.

Python 구조 계산 코드를 분석하여 다음 JSON을 반환하세요:
{
  "issues": [
    {
      "severity": "critical" | "warning" | "info",
      "category": "bug" | "unit" | "convention" | "safety" | "kds",
      "line_content": "문제가 있는 코드 줄",
      "description": "문제 설명",
      "suggestion": "개선 제안"
    }
  ],
  "kds_compliance": {
    "clause_references": true/false (KDS 조항 번호 명시 여부),
    "unit_consistency": true/false (SI 단위 일관성),
    "phi_parametrized": true/false (강도감소계수 매개변수화),
    "input_validation": true/false (입력값 검증),
    "type_hints": true/false (타입 힌트)
  },
  "overall_score": 1-10 (정수)
}

리뷰 기준:
1. 공학적 정확성: 수식, 단위, 계수가 KDS 기준에 맞는가
2. 코드 품질: 타입 힌트, 독스트링, 명명 규칙
3. 안전성: 입력 검증, 에러 처리, 경계값 확인
4. KDS 준수: 조항 참조, 강도감소계수 처리"""

    messages = [
        {
            "role": "user",
            "content": f"다음 구조 계산 Python 코드를 리뷰해주세요:\n\n```python\n{code}\n```"
        },
        {
            "role": "assistant",
            "content": "```json\n"
        }
    ]

    response = client.messages.create(
        model=model,
        max_tokens=2000,
        system=system,
        messages=messages,
        stop_sequences=["```"],
        temperature=0.0
    )

    return json.loads(response.content[0].text.strip())


# 리뷰 실행
review = review_structural_code(buggy_code)
print("=== 리뷰 결과 ===")
print(json.dumps(review, indent=2, ensure_ascii=False))

In [ ]:
# Step 2: 코드 개선 함수
def improve_structural_code(code: str, review_result: dict) -> str:
    """리뷰 결과를 반영하여 개선된 코드를 생성

    Args:
        code: 원본 코드
        review_result: review_structural_code()의 결과

    Returns:
        str: 개선된 Python 코드
    """
    system = """당신은 KDS 구조 설계기준 전문 Python 개발자입니다.

리뷰 결과를 반영하여 코드를 개선하세요.

개선 규칙:
1. 타입 힌트 추가 (from __future__ import annotations)
2. 독스트링에 KDS 조항 번호 명시
3. 단위를 SI (N, mm, MPa)로 통일
4. 강도감소계수(phi)를 매개변수로 추가
5. 입력값 검증 (ValueError) 추가
6. 반환값을 dict로 변경 (추적 가능)
7. print 대신 결과 dict에 판정 포함

개선된 Python 코드만 반환하세요."""

    messages = [
        {
            "role": "user",
            "content": (
                f"다음 코드를 리뷰 결과를 반영하여 개선해주세요.\n\n"
                f"=== 원본 코드 ===\n```python\n{code}\n```\n\n"
                f"=== 리뷰 결과 ===\n{json.dumps(review_result, indent=2, ensure_ascii=False)}"
            )
        },
        {
            "role": "assistant",
            "content": "```python\n"
        }
    ]

    response = client.messages.create(
        model=model,
        max_tokens=3000,
        system=system,
        messages=messages,
        stop_sequences=["```"],
        temperature=0.0
    )

    return response.content[0].text.strip()


# 개선 실행
improved = improve_structural_code(buggy_code, review)
print("=== 개선된 코드 ===")
print(improved)

In [ ]:
# Step 3: 개선된 코드를 다시 리뷰하여 점수 향상 확인
review_improved = review_structural_code(improved)

print(f"=== 리뷰 점수 비교 ===")
print(f"원본 코드: {review['overall_score']}/10")
print(f"개선 코드: {review_improved['overall_score']}/10")
print(f"\n=== KDS 준수 비교 ===")
for key in review["kds_compliance"]:
    orig = review["kds_compliance"][key]
    impr = review_improved["kds_compliance"].get(key, False)
    arrow = "->" if orig != impr else "=="
    print(f"  {key}: {orig} {arrow} {impr}")

In [ ]:
# Step 4: 검증 함수
def verify_exercise_1(review: dict, improved_code: str, review_improved: dict) -> None:
    """Exercise 1 결과를 검증"""
    # 원본 리뷰 검증
    assert "issues" in review, "리뷰에 issues가 없습니다"
    assert "kds_compliance" in review, "리뷰에 kds_compliance가 없습니다"
    assert "overall_score" in review, "리뷰에 overall_score가 없습니다"
    assert len(review["issues"]) >= 2, (
        f"최소 2개 이상의 이슈가 발견되어야 합니다: {len(review['issues'])}개"
    )

    # 원본 코드의 KDS 준수도는 낮아야 함
    kds = review["kds_compliance"]
    false_count = sum(1 for v in kds.values() if not v)
    assert false_count >= 3, (
        f"원본 코드의 KDS 비준수 항목이 3개 이상이어야 합니다: {false_count}개"
    )

    # 개선 코드 검증
    assert len(improved_code) > len(buggy_code), "개선 코드가 원본보다 길어야 합니다"
    assert "def " in improved_code, "개선 코드에 함수 정의가 있어야 합니다"

    # 점수 향상 검증
    assert review_improved["overall_score"] > review["overall_score"], (
        f"개선 후 점수가 올라야 합니다: "
        f"{review['overall_score']} -> {review_improved['overall_score']}"
    )

    print(f"PASS: 원본 이슈 {len(review['issues'])}건, "
          f"점수 {review['overall_score']} -> {review_improved['overall_score']}, "
          f"KDS 비준수 {false_count}건 개선")


verify_exercise_1(review, improved, review_improved)

---
## Exercise 2: 건설 시방서 자동 요약 시스템

### 배경
건설 시방서(Specification)는 공사의 기술적 요구사항을 상세히 기술한 문서입니다.
수십~수백 페이지에 달하는 시방서를 빠르게 파악하려면 구조화된 요약이 필요합니다.

### 문제
건설 시방서 텍스트를 입력받아 구조화된 JSON 요약을 생성하는 시스템을 구현하세요.

### 요구사항
1. `summarize_specification()`: 시방서를 구조화된 JSON으로 요약
2. `extract_quality_criteria()`: 시방서에서 품질 기준만 추출
3. 두 시방서 섹션의 요약을 비교하는 기능

### 기대 출력
```json
{
  "section_code": "05120",
  "section_name": "구조용 강재",
  "scope": "...",
  "key_requirements": ["..."],
  "referenced_standards": ["KS D 3503", ...],
  "quality_criteria": [{"item": "...", "standard": "...", "threshold": "..."}],
  "inspection_methods": ["..."]
}
```

In [ ]:
# ===== Exercise 2: 시방서 자동 요약 =====

# 샘플 시방서 텍스트 (구조용 강재)
spec_steel = """
05120 구조용 강재

1. 일반사항
1.1 적용 범위
본 시방은 건축 구조물에 사용되는 구조용 강재의
재료, 가공, 설치에 대한 요구사항을 규정한다.
기둥, 보, 가새 등 주요 구조 부재에 적용한다.

1.2 관련 기준
- KS D 3503 일반구조용 압연강재 (SS400)
- KS D 3515 용접구조용 압연강재 (SM490, SM520)
- KS B 0801 금속 재료 인장 시험법
- KS B 0896 강 용접 이음부의 초음파 탐상 시험법

2. 재료
2.1 구조용 강재는 KS D 3503 (SS400) 또는 KS D 3515 (SM490)에
적합한 것을 사용한다.
2.2 항복강도: SS400 — 235 MPa 이상, SM490 — 315 MPa 이상
2.3 인장강도: SS400 — 400-510 MPa, SM490 — 490-610 MPa
2.4 강재 밀 시트(Mill Sheet)를 제출하여 승인을 받아야 한다.

3. 가공
3.1 절단은 자동 가스 절단 또는 기계 절단으로 한다.
3.2 절단면의 거칠기는 50S 이하로 한다.
3.3 볼트 구멍은 드릴 가공을 원칙으로 한다. 펀칭은 판두께 12mm 이하에서만 허용.

4. 용접
4.1 용접사는 KS B 0885에 따른 자격을 보유해야 한다.
4.2 용접 절차서(WPS)를 사전에 승인받아야 한다.
4.3 용접봉은 KS D 7004에 적합한 것을 사용한다.

5. 품질 관리
5.1 용접부 비파괴검사: 완전용입 이음부 전수 UT 검사
    합격 기준: KS B 0896 2급 이상
5.2 볼트 체결: 설계 볼트 장력의 ±10% 이내
5.3 부재 치수 허용오차: KS D 3503 표 6에 따른다.
5.4 도장: KS M 6030에 따른 에폭시 프라이머
    건조막 두께 75μm 이상, 총 도막 두께 200μm 이상
""".strip()

# 샘플 시방서 텍스트 (콘크리트 공사)
spec_concrete = """
03300 현장치기 콘크리트

1. 일반사항
1.1 적용 범위
본 시방은 건축 구조물의 현장치기 콘크리트 공사에 대한
재료, 배합, 타설, 양생에 대한 요구사항을 규정한다.

1.2 관련 기준
- KS F 2405 콘크리트 압축강도 시험
- KS F 2421 콘크리트의 슬럼프 시험
- KDS 14 20 10 콘크리트구조 재료 기준

2. 재료
2.1 시멘트: KS L 5201 포틀랜드 시멘트 1종
2.2 설계기준강도(fck): 구조도면에 명시된 강도 (일반: 24-30 MPa)
2.3 굵은골재 최대치수: 25mm (일반), 부재 최소치수의 1/4 이하
2.4 물-시멘트비: 0.55 이하 (내구성 확보)

3. 타설
3.1 타설 시 자유낙하 높이: 1.5m 이하
3.2 1층 타설 두께: 600mm 이하
3.3 타설 후 진동다짐 실시 (내부진동기 간격 600mm 이하)
3.4 이어치기 시간 간격: 외기온 25°C 이상 시 90분 이내

4. 양생
4.1 습윤양생 기간: 최소 5일 (보통 포틀랜드 시멘트)
4.2 거푸집 존치 기간: 기둥/벽 — fck의 50% 도달 시
                     보/슬래브 — fck의 80% 도달 시

5. 품질 관리
5.1 압축강도 시험: 150일마다 1조(3개), KS F 2405
    합격 기준: 연속 3회 평균 >= fck, 개별 >= 0.85 * fck
5.2 슬럼프 시험: 매 운반차마다, 허용 ±25mm
5.3 공기량 시험: 4.5% ± 1.5%
5.4 염화물 함량: 0.3 kg/m3 이하
""".strip()

print(f"강재 시방서: {len(spec_steel)} 문자")
print(f"콘크리트 시방서: {len(spec_concrete)} 문자")

In [ ]:
# Step 1: 시방서 구조화 요약 함수
def summarize_specification(spec_text: str) -> dict:
    """시방서 텍스트를 구조화된 JSON으로 요약

    Args:
        spec_text: 시방서 원문 텍스트

    Returns:
        dict: 구조화된 요약
    """
    system = """당신은 건설 시방서 분석 전문가입니다.

시방서를 분석하여 다음 JSON을 반환하세요:
{
  "section_code": "시방서 섹션 코드 (예: 05120)",
  "section_name": "섹션 이름",
  "scope": "적용 범위 요약 (2문장 이내)",
  "key_requirements": ["핵심 요구사항 3-5개"],
  "referenced_standards": ["참조 표준 코드 (KS, KDS 등)"],
  "quality_criteria": [
    {"item": "검사 항목", "standard": "적용 기준", "threshold": "합격 기준값"}
  ],
  "inspection_methods": ["검사/시험 방법"]
}

규칙:
- 수치 기준은 원문 그대로 인용
- referenced_standards는 코드만 (예: "KS D 3503")
- quality_criteria는 정량적 기준만 포함"""

    messages = [
        {
            "role": "user",
            "content": f"다음 시방서를 구조화하여 요약해주세요:\n\n{spec_text}"
        },
        {
            "role": "assistant",
            "content": "```json\n"
        }
    ]

    response = client.messages.create(
        model=model,
        max_tokens=2000,
        system=system,
        messages=messages,
        stop_sequences=["```"],
        temperature=0.0
    )

    return json.loads(response.content[0].text.strip())


# 강재 시방서 요약
summary_steel = summarize_specification(spec_steel)
print("=== 강재 시방서 요약 ===")
print(json.dumps(summary_steel, indent=2, ensure_ascii=False))

In [ ]:
# Step 2: 콘크리트 시방서 요약
summary_concrete = summarize_specification(spec_concrete)
print("=== 콘크리트 시방서 요약 ===")
print(json.dumps(summary_concrete, indent=2, ensure_ascii=False))

In [ ]:
# Step 3: 품질 기준 추출 함수
def extract_quality_criteria(spec_text: str) -> list[dict]:
    """시방서에서 정량적 품질 기준만 추출

    Args:
        spec_text: 시방서 텍스트

    Returns:
        list: 품질 기준 리스트
    """
    system = """당신은 건설 품질 관리 전문가입니다.

시방서에서 정량적 품질 기준만 추출하여 JSON 배열로 반환하세요.

각 항목:
{
  "category": "재료" | "시공" | "검사" | "품질",
  "item": "검사/시험 항목",
  "standard": "적용 기준 코드",
  "threshold": "합격 기준 (수치 포함)",
  "frequency": "검사 빈도 (있는 경우)"
}

정성적 기준은 제외하고, 수치가 포함된 기준만 추출하세요."""

    messages = [
        {
            "role": "user",
            "content": f"다음 시방서에서 정량적 품질 기준을 추출해주세요:\n\n{spec_text}"
        },
        {
            "role": "assistant",
            "content": "```json\n"
        }
    ]

    response = client.messages.create(
        model=model,
        max_tokens=2000,
        system=system,
        messages=messages,
        stop_sequences=["```"],
        temperature=0.0
    )

    return json.loads(response.content[0].text.strip())


# 콘크리트 시방서 품질 기준 추출
quality = extract_quality_criteria(spec_concrete)
print(f"=== 콘크리트 시방서 품질 기준 ({len(quality)}건) ===")
for i, q in enumerate(quality, 1):
    print(f"  {i}. [{q.get('category', 'N/A')}] {q['item']}: {q['threshold']}")

In [ ]:
# Step 4: 검증 함수
def verify_exercise_2(
    summary_steel: dict,
    summary_concrete: dict,
    quality: list
) -> None:
    """Exercise 2 결과를 검증"""
    # 시방서 요약 공통 검증
    for name, summary in [("강재", summary_steel), ("콘크리트", summary_concrete)]:
        required_keys = [
            "section_code", "section_name", "scope",
            "key_requirements", "referenced_standards",
            "quality_criteria", "inspection_methods"
        ]
        for key in required_keys:
            assert key in summary, f"{name} 요약에 {key}가 없습니다"

        assert len(summary["key_requirements"]) >= 2, (
            f"{name}: 핵심 요구사항이 2개 이상이어야 합니다"
        )
        assert len(summary["referenced_standards"]) >= 2, (
            f"{name}: 참조 표준이 2개 이상이어야 합니다"
        )
        assert len(summary["quality_criteria"]) >= 1, (
            f"{name}: 품질 기준이 1개 이상이어야 합니다"
        )

    # 섹션 코드 확인
    assert "05120" in summary_steel["section_code"], "강재 섹션 코드가 05120이 아닙니다"
    assert "03300" in summary_concrete["section_code"], "콘크리트 섹션 코드가 03300이 아닙니다"

    # 품질 기준 추출 검증
    assert len(quality) >= 3, f"품질 기준이 3개 이상이어야 합니다: {len(quality)}개"
    for q in quality:
        assert "item" in q, "품질 기준에 item이 없습니다"
        assert "threshold" in q, "품질 기준에 threshold가 없습니다"

    print(f"PASS: 강재 요약 (요구사항 {len(summary_steel['key_requirements'])}건, "
          f"표준 {len(summary_steel['referenced_standards'])}건), "
          f"콘크리트 요약 (요구사항 {len(summary_concrete['key_requirements'])}건), "
          f"품질 기준 {len(quality)}건")


verify_exercise_2(summary_steel, summary_concrete, quality)

---
## Exercise 3: 계약서 비교 분석

### 배경
건설 계약서의 수정본이 발행되면, 원본과의 차이를 정확히 파악해야 합니다.
특히 공사대금, 공기, 지체상금 등의 변경은 **위험 요소**가 될 수 있습니다.

### 문제
두 버전의 건설 계약서를 비교하여, 변경된 조항과 위험도를 분석하는 시스템을 구현하세요.

### 요구사항
1. `compare_contracts()`: 두 계약서를 비교하여 변경 사항을 JSON으로 반환
2. `assess_risk()`: 변경 사항의 위험도를 평가
3. 각 변경에 대한 조치 권고 포함

### 기대 출력
```json
{
  "changes": [
    {
      "clause": "제5조 1항",
      "type": "modified",
      "original": "금 50억원",
      "revised": "금 55억원",
      "risk_level": "medium",
      "recommendation": "..."
    }
  ],
  "summary": {"added": 1, "modified": 3, "removed": 0},
  "overall_risk": "medium"
}
```

In [ ]:
# ===== Exercise 3: 계약서 비교 분석 =====

# 샘플 계약서 (원본)
contract_v1 = """
건설공사 도급계약서 (원본, 2026.01.15)

제1조 (목적)
본 계약은 ○○아파트 신축공사의 도급에 관한 사항을 정한다.

제2조 (공사 범위)
구조, 건축, 기계설비, 전기설비 일체를 포함한다.
지하 2층, 지상 20층, 3개동, 연면적 45,000m2

제3조 (공사대금)
1. 총 공사대금은 금 500억원으로 한다.
2. 기성금은 매월 말 기성 비율에 따라 지급한다.
3. 선급금은 계약금액의 20%로 한다.

제4조 (공사기간)
1. 착공일: 2026년 4월 1일
2. 준공일: 2028년 3월 31일 (24개월)
3. 기간 연장은 천재지변, 설계변경 등 정당한 사유 시
   발주자 승인 후 가능하다.

제5조 (지체상금)
1. 지체상금률: 1일당 계약금액의 1/1000
2. 최대 지체상금: 계약금액의 10%

제6조 (하자보수)
1. 구조체: 10년
2. 방수: 5년
3. 마감: 2년

제7조 (분쟁 해결)
1. 분쟁은 대한상사중재원의 중재로 해결한다.
""".strip()

# 샘플 계약서 (수정본)
contract_v2 = """
건설공사 도급계약서 (수정본, 2026.03.01)

제1조 (목적)
본 계약은 ○○아파트 신축공사의 도급에 관한 사항을 정한다.

제2조 (공사 범위)
구조, 건축, 기계설비, 전기설비 일체를 포함한다.
지하 2층, 지상 25층, 3개동, 연면적 52,000m2

제3조 (공사대금)
1. 총 공사대금은 금 580억원으로 한다.
2. 기성금은 매월 말 기성 비율에 따라 지급하되,
   검수 완료 후 30일 이내에 지급한다.
3. 선급금은 계약금액의 15%로 한다.

제4조 (공사기간)
1. 착공일: 2026년 4월 1일
2. 준공일: 2028년 9월 30일 (30개월)
3. 기간 연장은 천재지변, 설계변경 등 정당한 사유 시
   발주자 승인 후 가능하다.

제5조 (지체상금)
1. 지체상금률: 1일당 계약금액의 1/1500
2. 최대 지체상금: 계약금액의 7%

제6조 (하자보수)
1. 구조체: 10년
2. 방수: 5년
3. 마감: 2년

제7조 (설계변경)
1. 공사 중 설계변경이 필요한 경우 변경 내역을 서면으로 통보한다.
2. 설계변경에 따른 공사대금 조정은 실비 정산을 원칙으로 한다.
3. 설계변경 승인 전 시공분에 대해서는 수급인이 책임진다.

제8조 (분쟁 해결)
1. 분쟁은 서울중앙지방법원을 관할법원으로 한다.
""".strip()

print(f"원본 계약서: {len(contract_v1)} 문자")
print(f"수정본 계약서: {len(contract_v2)} 문자")

In [ ]:
# Step 1: 계약서 비교 함수
def compare_contracts(original: str, revised: str) -> dict:
    """두 계약서를 비교하여 변경 사항을 분석

    Args:
        original: 원본 계약서 텍스트
        revised: 수정본 계약서 텍스트

    Returns:
        dict: 비교 분석 결과
    """
    system = """당신은 건설 계약 분석 전문가입니다.

두 계약서를 조항별로 비교하여 다음 JSON을 반환하세요:
{
  "changes": [
    {
      "clause": "변경된 조항 (예: 제3조 1항)",
      "type": "added" | "modified" | "removed",
      "original": "원본 내용 (modified/removed인 경우)",
      "revised": "수정 내용 (added/modified인 경우)",
      "risk_level": "low" | "medium" | "high",
      "risk_description": "위험 요인 설명",
      "recommendation": "조치 권고"
    }
  ],
  "summary": {
    "added": 숫자,
    "modified": 숫자,
    "removed": 숫자
  },
  "overall_risk": "low" | "medium" | "high"
}

위험도 판단 기준:
- high: 공사대금 10% 이상 변경, 공기 6개월 이상 연장, 지체상금률 대폭 변경
- medium: 공사대금 5-10% 변경, 공기 3-6개월 연장, 신규 조항 추가
- low: 경미한 문구 수정, 동일 내용 유지"""

    messages = [
        {
            "role": "user",
            "content": (
                f"다음 두 계약서를 비교 분석해주세요.\n\n"
                f"=== 원본 계약서 ===\n{original}\n\n"
                f"=== 수정본 계약서 ===\n{revised}"
            )
        },
        {
            "role": "assistant",
            "content": "```json\n"
        }
    ]

    response = client.messages.create(
        model=model,
        max_tokens=3000,
        system=system,
        messages=messages,
        stop_sequences=["```"],
        temperature=0.0
    )

    return json.loads(response.content[0].text.strip())


# 비교 실행
comparison = compare_contracts(contract_v1, contract_v2)
print("=== 계약서 비교 결과 ===")
print(json.dumps(comparison, indent=2, ensure_ascii=False))

In [ ]:
# Step 2: 변경 사항 위험도별 요약 출력
def print_risk_summary(comparison: dict) -> None:
    """비교 결과를 위험도별로 정리하여 출력"""
    changes = comparison["changes"]
    summary = comparison["summary"]

    print(f"=" * 60)
    print(f"계약서 변경 분석 보고서")
    print(f"=" * 60)
    print(f"\n변경 현황: 추가 {summary.get('added', 0)}건 | "
          f"수정 {summary.get('modified', 0)}건 | "
          f"삭제 {summary.get('removed', 0)}건")
    print(f"종합 위험도: {comparison.get('overall_risk', 'N/A').upper()}")

    # 위험도별 분류
    for level in ["high", "medium", "low"]:
        level_changes = [c for c in changes if c.get("risk_level") == level]
        if level_changes:
            label = {"high": "[위험]", "medium": "[주의]", "low": "[참고]"}[level]
            print(f"\n--- {label} ({len(level_changes)}건) ---")
            for c in level_changes:
                print(f"  {c['clause']} ({c['type']})")
                if c.get("original"):
                    print(f"    원본: {c['original']}")
                if c.get("revised"):
                    print(f"    수정: {c['revised']}")
                if c.get("recommendation"):
                    print(f"    권고: {c['recommendation']}")


print_risk_summary(comparison)

In [ ]:
# Step 3: 위험 항목에 대한 대응 전략 생성
def generate_response_strategy(comparison: dict) -> dict:
    """고위험 변경 사항에 대한 대응 전략 생성

    Args:
        comparison: compare_contracts()의 결과

    Returns:
        dict: 대응 전략
    """
    # 고위험 + 중위험 항목만 추출
    risky_changes = [
        c for c in comparison["changes"]
        if c.get("risk_level") in ("high", "medium")
    ]

    if not risky_changes:
        return {"strategy": "no_action_needed", "items": []}

    system = """당신은 건설 계약 협상 전문가입니다.

고위험/중위험 계약 변경 사항에 대한 대응 전략을 JSON으로 작성하세요:
{
  "overall_strategy": "전체 대응 방향 (2문장)",
  "items": [
    {
      "clause": "조항",
      "action": "accept" | "negotiate" | "reject",
      "negotiation_points": ["협상 포인트"],
      "fallback_position": "최소 수용 기준"
    }
  ],
  "priority_order": ["협상 우선순위 (조항 번호)"]
}"""

    messages = [
        {
            "role": "user",
            "content": (
                "다음 계약 변경 사항에 대한 대응 전략을 수립해주세요:\n\n"
                + json.dumps(risky_changes, indent=2, ensure_ascii=False)
            )
        },
        {
            "role": "assistant",
            "content": "```json\n"
        }
    ]

    response = client.messages.create(
        model=model,
        max_tokens=2000,
        system=system,
        messages=messages,
        stop_sequences=["```"],
        temperature=0.0
    )

    return json.loads(response.content[0].text.strip())


strategy = generate_response_strategy(comparison)
print("=== 대응 전략 ===")
print(json.dumps(strategy, indent=2, ensure_ascii=False))

In [ ]:
# Step 4: 검증 함수
def verify_exercise_3(comparison: dict, strategy: dict) -> None:
    """Exercise 3 결과를 검증"""
    # 비교 결과 구조 검증
    assert "changes" in comparison, "comparison에 changes가 없습니다"
    assert "summary" in comparison, "comparison에 summary가 없습니다"
    assert "overall_risk" in comparison, "comparison에 overall_risk가 없습니다"

    # 변경 사항 개수 확인
    changes = comparison["changes"]
    assert len(changes) >= 3, f"최소 3개 이상의 변경이 있어야 합니다: {len(changes)}개"

    # 각 변경 사항 구조 확인
    valid_types = {"added", "modified", "removed"}
    valid_risks = {"low", "medium", "high"}
    for c in changes:
        assert "clause" in c, "변경 사항에 clause가 없습니다"
        assert "type" in c, "변경 사항에 type이 없습니다"
        assert c["type"] in valid_types, f"유효하지 않은 type: {c['type']}"
        assert "risk_level" in c, "변경 사항에 risk_level이 없습니다"
        assert c["risk_level"] in valid_risks, f"유효하지 않은 risk_level: {c['risk_level']}"

    # summary 확인
    summary = comparison["summary"]
    total = summary.get("added", 0) + summary.get("modified", 0) + summary.get("removed", 0)
    assert total >= 3, f"summary 합계가 3 이상이어야 합니다: {total}"

    # 대응 전략 검증
    if strategy.get("items"):
        valid_actions = {"accept", "negotiate", "reject"}
        for item in strategy["items"]:
            assert "clause" in item, "전략 항목에 clause가 없습니다"
            assert "action" in item, "전략 항목에 action이 없습니다"
            assert item["action"] in valid_actions, (
                f"유효하지 않은 action: {item['action']}"
            )

    # 공사대금 변경이 감지되었는지 확인
    amount_changes = [
        c for c in changes
        if "대금" in c.get("clause", "") or "대금" in c.get("original", "")
        or "대금" in c.get("revised", "") or "금액" in str(c)
        or "3조" in c.get("clause", "")
    ]
    assert len(amount_changes) >= 1, "공사대금 변경이 감지되어야 합니다"

    high_risk = [c for c in changes if c["risk_level"] == "high"]
    print(f"PASS: 변경 {len(changes)}건 (추가 {summary.get('added', 0)}, "
          f"수정 {summary.get('modified', 0)}, 삭제 {summary.get('removed', 0)}), "
          f"고위험 {len(high_risk)}건, "
          f"종합 위험도: {comparison['overall_risk']}, "
          f"대응 전략 {len(strategy.get('items', []))}건")


verify_exercise_3(comparison, strategy)

---
## 핵심 정리

| 실습 | 핵심 기법 | 건축공학 적용 |
|------|-----------|---------------|
| **구조 코드 리뷰** | 시스템 프롬프트 + JSON 추출 | KDS 준수 여부 자동 검사, 코드 품질 점수화 |
| **시방서 요약** | 프리필링 + 정지 시퀀스 | 섹션별 구조화, 품질 기준 정량 추출 |
| **계약서 비교** | 멀티턴 분석 + 위험도 평가 | 조항별 변경 감지, 위험 등급화, 대응 전략 |

### Claude Code와의 연결

이 노트북에서 구현한 함수들을 Claude Code 프로젝트에 통합하면:

1. **CLAUDE.md**에 리뷰 규칙을 명시 → Claude Code가 자동으로 KDS 준수 코드 생성
2. **Skills**로 `/spec-summary` 명령 등록 → 자연어로 시방서 요약 호출
3. **Hooks**로 PostWrite 시 자동 코드 리뷰 → 저장할 때마다 품질 검사
4. **Supabase**에 분석 결과 저장 → 팀원과 공유, 이력 관리